# Applying quantization to an MNIST model

In this tutorial, we will be providing a basic introduction to quantizing a model with CoreAI-Opt.

After the end of this tutorial, you should be familiar with the following:
1. [How to apply CoreAI-Opt's Weight-Only quantization](#Weight-Only-Quantization)
2. [How to apply CoreAI-Opt's Weight + Activation quantization](#Weight-and-Activation-Quantization)
3. [How to apply CoreAI-Opt's Quantization-Aware Training](#Quantization-Aware-Training)
4. [How to export CoreAI-Opt quantized models to Core AI](#Export-to-Core-AI)

**Table of Contents:**

- [Setup](#Setup)
  - [MNIST Dataset download](#MNIST-Dataset-download)
  - [Model definition](#Model-definition)
  - [Training and Evaluation](#Training-and-Evaluation)
  - [Train unquantized model](#Train-unquantized-model)
- [Weight Only Quantization](#Weight-Only-Quantization)
- [Weight and Activation Quantization](#Weight-and-Activation-Quantization)
- [Quantization Aware Training](#Quantization-Aware-Training)
- [Export to Core AI](#Export-to-Core-AI)

## Setup

We will be using a basic CNN model and train it on the MNIST dataset and observe its final accuracy.

Once we train this CNN model, we will apply quantization to it using `coreai-opt` starting with *weight only quantization (data free)*, then moving to apply calibration data based *weight + activation quantization*, and then finally do *Quantization Aware Training (QAT)*.

In [26]:
import random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchinfo import summary
from torchvision import datasets, transforms

In [27]:
from coreai_torch import (
    TorchConverter,
    ExternalizeSpec,
    mark_for_externalization,
    get_decomp_table,
)


In [28]:
SEED = 1976

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [29]:
# Used to save intermediate results and datasets
SAVE_DIRECTORY = "."

### MNIST Dataset download

Helper to download the MNIST dataset with standard normalization applied.

In [30]:
def mnist_transforms() -> transforms.Compose:
    return transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])


def download_mnist_dataset(
    download_path: Path, transform: transforms.Compose | None = None
) -> tuple[datasets.MNIST, datasets.MNIST]:
    if transform is None:
        transform = mnist_transforms()
    train = datasets.MNIST(download_path, train=True, download=True, transform=transform)
    test = datasets.MNIST(download_path, train=False, download=True, transform=transform)
    return train, test

### Model definition

A simple CNN with a single Conv2d → ReLU → MaxPool block, followed by Flatten and a Linear classifier.

In [31]:
from coreai_torch.composite_ops import RMSNormImpl
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        # self.eps = eps
        self.norm = RMSNormImpl(eps=eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm(x, self.weight)


class MnistNetwork(nn.Module):
    def __init__(self, num_classes: int = 10, state_dict: dict | None = None) -> None:
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 12, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, stride=2, padding=0),
            nn.Flatten(),
            RMSNorm(2352),
            nn.Linear(2352, num_classes),
        )
        if state_dict is not None:
            self.load_state_dict(state_dict)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

### Training and Evaluation

Standard PyTorch training loop and evaluation function that computes accuracy.

In [32]:
def train_step(model, optimizer, loss_fn, inputs, ground_truth) -> float:
    model.train()
    device = next(model.parameters()).device
    inputs = inputs.to(device)
    ground_truth = ground_truth.to(device)
    optimizer.zero_grad()
    predictions = model(inputs)
    loss = loss_fn(predictions, ground_truth)
    loss.backward()
    optimizer.step()
    return loss.item()


def train_epoch(model, train_loader, optimizer, loss_fn) -> float:
    total_loss = 0.0
    for inputs, ground_truth in train_loader:
        loss = train_step(model, optimizer, loss_fn, inputs, ground_truth)
        total_loss += loss
    return total_loss / len(train_loader)


def create_adam_optimizer(model: nn.Module, lr: float = 1e-3) -> torch.optim.Adam:
    return torch.optim.Adam(model.parameters(), lr=lr)


def eval_model(model: nn.Module, test_dataloader: DataLoader) -> float:
    model.eval()
    device = next(model.parameters()).device
    num_correct = 0
    total = 0
    with torch.no_grad():
        for inputs, ground_truth in test_dataloader:
            inputs = inputs.to(device)
            ground_truth = ground_truth.to(device)
            predictions = model(inputs)
            _, predicted = torch.max(predictions.data, 1)
            total += ground_truth.size(0)
            num_correct += (predicted == ground_truth).sum().item()
    return num_correct / total

In [33]:
# Download and instantiate datasets
DOWNLOAD_PATH = Path(SAVE_DIRECTORY) / ".mnist_dataset"

train_dataset, test_dataset = download_mnist_dataset(
    download_path=DOWNLOAD_PATH
)

In [34]:
BATCH_SIZE = 128

# Instantiate dataloaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE)

The CNN model used for this tutorial contains a single Conv2d, ReLU, MaxPool2d, Flatten, and Linear layer. Here's the structure:

In [35]:
basic_cnn_model = MnistNetwork(num_classes=10)

print(basic_cnn_model)
# Print summary of model
# summary(basic_cnn_model, input_size=(1, 1, 28, 28))

MnistNetwork(
  (model): Sequential(
    (0): Conv2d(1, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Flatten(start_dim=1, end_dim=-1)
    (4): RMSNorm(
      (norm): RMSNormImpl()
    )
    (5): Linear(in_features=2352, out_features=10, bias=True)
  )
)


### Train unquantized model

Let's train this model (unquantized) so we can get a baseline accuracy. We save the trained weights so we can reload them for each quantization experiment.

In [36]:
EPOCHS = 1

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = create_adam_optimizer(basic_cnn_model)

basic_cnn_model = basic_cnn_model.to("mps")

epoch_results = []
for epoch in range(EPOCHS):
    epoch_avg_loss = train_epoch(
        model=basic_cnn_model, train_loader=train_loader, optimizer=optimizer, loss_fn=loss_fn
    )
    epoch_results.append(f"  Epoch {epoch + 1}: loss={epoch_avg_loss:.4f}")

basic_cnn_model = basic_cnn_model.cpu().eval()
print("\n".join(epoch_results))

# Save trained weights for reuse in quantization sections
pretrained_state_dict = basic_cnn_model.state_dict()

  Epoch 1: loss=0.2546


In [37]:
accuracy = eval_model(basic_cnn_model, test_loader)
print(f"Baseline accuracy: {accuracy:.4f}")

Baseline accuracy: 0.9688


## Weight Only Quantization

Weight-only quantization compresses model weights to a lower precision (e.g., INT8) while keeping activations in full precision (FP32). This is the simplest form of quantization — it requires **no calibration data** and can be applied directly to a trained model.

In [38]:
from coreai_opt.quantization import (
    ModuleQuantizerConfig,
    QuantizationSpec,
    Quantizer,
    QuantizerConfig,
)

example_inputs = (torch.randn(1, 1, 28, 28),)

For this tutorial, we will use the `INT8` dtype. Refer to the `QuantizationSpec` reference for all options.

In [39]:
WEIGHT_DTYPE = "int8"

`QuantizerConfig` describes how to quantize each operation through three spec keys:

- `op_state_spec`: state tensors of the op (weights, biases).
- `op_input_spec`: input activations — tensors flowing into the op.
- `op_output_spec`: output activations — tensors flowing out of the op.

For weight-only quantization, only the `weight` state spec is populated; setting `op_input_spec` and `op_output_spec` to `None` tells the quantizer to leave activations in full precision.

In [40]:
def show_externalize_state(model, label):
      print(f"\n========== {label} ==========")

      # 1. Per-submodule marker attributes + forward swap
      found = False
      for name, mod in model.named_modules():
          attrs = {
              a: getattr(mod, a)
              for a in ("_externalize_name", "_externalize_op_name",
                        "_externalize_config", "_original_forward")
              if hasattr(mod, a)
          }
          if not attrs:
              continue
          found = True
          print(f"\n  submodule [{name}]  type={type(mod).__name__}")
          for k, v in attrs.items():
              print(f"    {k} = {v!r}")
          # forward identity is the clearest signal of the patch
          print(f"    forward         = {mod.forward!r}")
          print(f"    forward.__name__= {getattr(mod.forward, '__name__', '<?>')}")
          print(f"    forward.__module__={getattr(mod.forward, '__module__', '<?>')}")

      if not found:
          print("  (no submodules carry externalize markers)")

      # 2. Global torch.library registry under our namespace
      ext_ns = getattr(torch.ops, "coreai_torch_ext", None)
      print(f"Ext_ns: {ext_ns}")
      if ext_ns is None:
          print("\n  torch.ops.coreai_torch_ext: <namespace not registered>")
      else:
          registered = [n for n in dir(ext_ns) if not n.startswith("_")]
          print(f"\n  torch.ops.coreai_torch_ext ops: {registered}")

In [41]:
wo_model = MnistNetwork(num_classes=10, state_dict=pretrained_state_dict)
wo_model.eval()

weight_spec = QuantizationSpec(dtype=WEIGHT_DTYPE)

wo_config = QuantizerConfig(
    global_config=ModuleQuantizerConfig(
        op_state_spec={"weight": weight_spec},
        op_input_spec=None,
        op_output_spec=None,
    )
)

show_externalize_state(wo_model, "BEFORE ")
markers = mark_for_externalization(
    wo_model,
    [
        ExternalizeSpec(
            target_class=RMSNormImpl,
            composite_op_name="rms_norm",
            composite_attrs=["axes", "eps"],
        ),
    ],
)

show_externalize_state(wo_model, "AFTER ")


wo_quantizer = Quantizer(wo_model, wo_config)
wo_prepared = wo_quantizer.prepare(example_inputs)
print(f"Prepared weight-only quantization with dtype={WEIGHT_DTYPE}")


========== BEFORE  ==========
  (no submodules carry externalize markers)
Ext_ns: <module 'torch.ops.coreai_torch_ext' from 'torch.ops'>

  torch.ops.coreai_torch_ext ops: ['model_4_norm', 'name']

========== AFTER  ==========

  submodule [model.4.norm]  type=RMSNormImpl
    _externalize_name = 'model.4.norm'
    _externalize_op_name = 'model_4_norm'
    _externalize_config = ExternalizeSpec(target_class=<class 'coreai_torch.composite_ops._rms_norm.RMSNormImpl'>, composite_op_name='rms_norm', composite_attrs=['axes', 'eps'])
    _original_forward = <bound method RMSNormImpl.forward of RMSNormImpl()>
    forward         = <function _prepare_module.<locals>.patched_forward at 0x1393e6e80>
    forward.__name__= patched_forward
    forward.__module__=coreai_torch.externalize
Ext_ns: <module 'torch.ops.coreai_torch_ext' from 'torch.ops'>

  torch.ops.coreai_torch_ext ops: ['model_4_norm', 'name']


/Users/pkmandke/.local/share/uv/python/cpython-3.11.14-macos-aarch64-none/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Prepared weight-only quantization with dtype=int8


> Note: `coreai-opt` also offers `QuantizerConfig.presets.w8()` as a shortcut for the above config.

After calling `prepare()`, the model now has FakeQuantize modules inserted for weight tensors. These simulate quantization during the forward pass.

In [42]:
wo_accuracy = eval_model(wo_prepared, test_loader)
print(f"Weight-only PTQ accuracy: {wo_accuracy:.4f}")

Weight-only PTQ accuracy: 0.9687


## Weight and Activation Quantization

Now, let's try using weight + activation quantization through `coreai-opt`. 

Weight + Activation quantization compresses both the model's weights **and** intermediate activation tensors to a lower precision. This provides a greater speedup than weight-only, but requires **calibration data** — representative inputs are run through the model so it can compute appropriate scale and zero-point values for the quantized activations.

For this tutorial, we will use the `INT8` dtype for activations. Refer to the `QuantizationSpec` reference for all options.

In [43]:
ACTIVATION_DTYPE = "int8"

Two changes from the weight-only config:

- `op_input_spec` and `op_output_spec` are now populated, so activations are quantized too.
- The `"*"` key applies the same INT8 spec to every operation input/output.

In [44]:
wa_model = MnistNetwork(num_classes=10, state_dict=pretrained_state_dict)
wa_model.eval()

activation_spec = QuantizationSpec(dtype=ACTIVATION_DTYPE)

wa_config = QuantizerConfig(
    global_config=ModuleQuantizerConfig(
        op_state_spec={"weight": weight_spec},
        op_input_spec={"*": activation_spec},
        op_output_spec={"*": activation_spec},
    )
)

markers = mark_for_externalization(
    wa_model,
    [
        ExternalizeSpec(
            target_class=RMSNormImpl,
            composite_op_name="rms_norm",
            composite_attrs=["axes", "eps"],
        ),
    ],
)
wa_quantizer = Quantizer(wa_model, wa_config)
wa_prepared = wa_quantizer.prepare(example_inputs)
print(
    f"Prepared weight + activation quantization (weight={WEIGHT_DTYPE}, activation={ACTIVATION_DTYPE})"
)

/Users/pkmandke/.local/share/uv/python/cpython-3.11.14-macos-aarch64-none/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Prepared weight + activation quantization (weight=int8, activation=int8)


We now call `calibration_mode()` and feed it representative inputs to populate scale and zero-point value.

The `calibration_mode()` context manager enables range observers (to collect activation statistics) while disabling fake quantization (so the forward pass is numerically identical to the unquantized model). After exiting the context, observers are frozen and fake quantization is re-enabled.

In [45]:
NUM_CALIBRATION_BATCHES = 1

with wa_quantizer.calibration_mode():
    for i, (data, _) in enumerate(train_loader):
        if i >= NUM_CALIBRATION_BATCHES:
            break
        wa_prepared(data)

print(
    f"Calibrated with {NUM_CALIBRATION_BATCHES} batch(es) ({NUM_CALIBRATION_BATCHES * BATCH_SIZE} samples)"
)

Calibrated with 1 batch(es) (128 samples)


In [46]:
wa_accuracy = eval_model(wa_prepared, test_loader)
print(f"Weight + Activation accuracy after prepare + calibration: {wa_accuracy:.4f}")

Weight + Activation accuracy after prepare + calibration: 0.9645


## Quantization Aware Training

Now, let's try to fine-tune the weight + activation quantized model with QAT to recover any accuracy lost from quantization.

The QAT training loop wraps each epoch in `wa_quantizer.training_mode()`:

- During the context: fake quantization is enabled and observers are enabled (collecting activation statistics during training batches).
- On exit: observers are frozen; fake quantization stays enabled, so the model is ready for evaluation with quant-aware weights.

In [47]:
QAT_EPOCHS = 10
qat_optimizer = create_adam_optimizer(wa_prepared, lr=1e-4)

wa_prepared.to("cpu")

for epoch in range(QAT_EPOCHS):
    with wa_quantizer.training_mode():
        epoch_loss = train_epoch(
            model=wa_prepared,
            train_loader=train_loader,
            optimizer=qat_optimizer,
            loss_fn=torch.nn.CrossEntropyLoss(),
        )

    qat_acc = eval_model(wa_prepared, test_loader)
    print(f"  Epoch {epoch + 1}: loss={epoch_loss:.4f}, accuracy={qat_acc:.4f}")

qat_prepared = wa_prepared.cpu()
print(f"\nQAT final accuracy: {qat_acc:.4f}")

  Epoch 1: loss=0.0905, accuracy=0.9745
  Epoch 2: loss=0.0808, accuracy=0.9762
  Epoch 3: loss=0.0741, accuracy=0.9788
  Epoch 4: loss=0.0686, accuracy=0.9793
  Epoch 5: loss=0.0641, accuracy=0.9799
  Epoch 6: loss=0.0603, accuracy=0.9794
  Epoch 7: loss=0.0572, accuracy=0.9806
  Epoch 8: loss=0.0541, accuracy=0.9818
  Epoch 9: loss=0.0519, accuracy=0.9814
  Epoch 10: loss=0.0494, accuracy=0.9828

QAT final accuracy: 0.9828


## Export to Core AI

Once the quantized model is ready, call `finalize()` to convert the fake quantization modules into the real quantized representation for deployment.

Pass `ExportBackend.CoreAI` to `finalize(backend=...)` to target the `.aimodel` format produced by `coreai-torch`.

We'll export the finetuned weight + activation model from the previous section.

In [48]:
from coreai_opt import ExportBackend

coreai_model = wa_quantizer.finalize(backend=ExportBackend.CoreAI)

The export proceeds in three steps:

- Trace the model with `torch.export.export()` to obtain a graph representation.
- Apply `cast_to_16_bit_precision()` to cast remaining FP32 parameters to FP16 for optimal on-device performance.
- Convert the exported program to Core AI format using `coreai-torch.TorchConverter`.

In [49]:
import shutil

from coreai_opt.casting import cast_to_16_bit_precision
from coreai_torch import TorchConverter, get_decomp_table

exported_program = torch.export.export(coreai_model, example_inputs, strict=False)
exported_program = exported_program.run_decompositions(get_decomp_table())
cast_to_16_bit_precision(exported_program)
print(exported_program)

markers.subexport_and_restore(exported_program)
coreai_program = TorchConverter().add_exported_program(exported_program, externalize_markers=markers).to_coreai()
coreai_program.optimize()

output_path = Path(SAVE_DIRECTORY) / "exported_model.aimodel"
if output_path.exists():
    shutil.rmtree(output_path)
coreai_program.save_asset(output_path)
print(f"Exported: {output_path}")

ExportedProgram:
    class GraphModule(torch.nn.Module):
        def forward(self, p_model_0_weight: "f16[12, 1, 3, 3]", p_model_0_bias: "f16[12]", p_model_4_weight: "f16[2352]", p_model_5_weight: "f16[10, 2352]", p_model_5_bias: "f16[10]", b_model_0_weight_scale: "f16[1, 1, 1, 1]", b_model_0_weight_zero_point: "i8[1, 1, 1, 1]", b_model_0_weight_quantized: "i8[12, 1, 3, 3]", b_model_5_weight_scale: "f16[1, 1]", b_model_5_weight_zero_point: "i8[1, 1]", b_model_5_weight_quantized: "i8[10, 2352]", b_activation_post_process_0_scale: "f16[]", b_activation_post_process_0_zero_point: "i8[]", b_activation_post_process_2_scale: "f16[]", b_activation_post_process_2_zero_point: "i8[]", b_activation_post_process_3_scale: "f16[]", b_activation_post_process_3_zero_point: "i8[]", b_activation_post_process_4_scale: "f16[]", b_activation_post_process_4_zero_point: "i8[]", b_activation_post_process_5_scale: "f16[]", b_activation_post_process_5_zero_point: "i8[]", b_activation_post_process_7_scale: "f16[

/Users/pkmandke/.local/share/uv/python/cpython-3.11.14-macos-aarch64-none/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/Users/pkmandke/.local/share/uv/python/cpython-3.11.14-macos-aarch64-none/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


coreai-torch 0.4.0: converting 1 program(s) to Core AI

Exported: exported_model.aimodel


In [50]:
print(coreai_program)

module {
  coreai.graph private noinline @model.4.norm_6e2877fa(%arg0: tensor<1x2352xf16> {coreai.name = "input"}, %arg1: tensor<2352xf16> {coreai.name = "scale"}) -> (tensor<1x2352xf16> {coreai.name = "mul_2"}) attributes {__coreai_pure__, composite_decl = #coreai.composite_declaration<"rms_norm" = {input_names = ["input", "scale"], op_attrs = {axes = -1 : si64, eps = 9.99999974E-6 : f32, version = 1 : si64}, output_names = ["output"]}>} {
    %0 = coreai.constant dense<1> : tensor<1xsi32>
    %1 = coreai.constant dense<9.99999974E-6> : tensor<f32>
    %2 = coreai.cast %arg0 : tensor<1x2352xf16> to tensor<1x2352xf32>
    %3 = coreai.decomposable.broadcasting_mul %2, %2 : (tensor<1x2352xf32>, tensor<1x2352xf32>) -> tensor<1x2352xf32>
    %4 = coreai.reduce_mean %3, %0 : (tensor<1x2352xf32>, tensor<1xsi32>) -> tensor<1x1xf32>
    %5 = coreai.decomposable.broadcasting_add %4, %1 : (tensor<1x1xf32>, tensor<f32>) -> tensor<1x1xf32>
    %6 = coreai.rsqrt %5 : tensor<1x1xf32> -> tensor<1x1xf